# Criticism Data Collection Runbook

This notebook helps collaborators run data collection without editing package code. It performs network collection when enabled, can run for many hours on multi-year GDELT DOC collection, and can be interrupted safely because successful DOC day payloads are checkpointed.

Raw outputs are never overwritten. Existing daily DOC checkpoints are reused. This notebook is a runbook, not core pipeline logic; the authoritative implementation remains in `src/mci/` and `scripts/`.

## Smoke-Test Parameters

Start here. A one-day run verifies credentials, network access, deterministic paths, checkpointing, and previews without launching a long collection.

In [ ]:
from datetime import date
from pathlib import Path

PROJECT_ROOT_OVERRIDE = None
START_DATE = date(2022, 1, 3)
END_DATE = date(2022, 1, 3)
MAX_RECORDS = 25
RUN_GDELT = True
RUN_MARKET_DATA = True
RESUME = True
OVERWRITE_INTERIM = False

REQUEST_PAUSE_SECONDS = 10
MAX_RETRIES = 10
BACKOFF_SECONDS = 30
MAX_BACKOFF_SECONDS = 1800
JITTER_SECONDS = 10

## Full MVP Date Range

After the smoke test succeeds, uncomment this cell and rerun the dry-run/status cells before starting collection. A full two-query DOC run may take many hours.

In [ ]:
# START_DATE = date(2022, 1, 1)
# END_DATE = date(2026, 5, 31)
# MAX_RECORDS = 250
# RUN_GDELT = True
# RUN_MARKET_DATA = True
# RESUME = True
# OVERWRITE_INTERIM = False

## Import And Setup

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd


def find_project_root(start: Path) -> Path:
    candidates = []
    project_root_override = globals().get("PROJECT_ROOT_OVERRIDE")
    if project_root_override:
        candidates.append(Path(project_root_override).expanduser())

    env_project_root = os.environ.get("MCI_PROJECT_ROOT")
    if env_project_root:
        candidates.append(Path(env_project_root).expanduser())

    candidates.extend([start, *start.parents])
    candidates.extend([
        start / "market-criticism-index",
        Path.home() / "Desktop" / "market-criticism-index",
        Path.home() / "market-criticism-index",
    ])

    seen_candidates = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen_candidates:
            continue
        seen_candidates.add(candidate)
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "mci").exists():
            return candidate
    raise RuntimeError("Could not find the project root. Start Jupyter from the repository checkout, set PROJECT_ROOT_OVERRIDE in the Parameters cell, or set MCI_PROJECT_ROOT.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from mci.gdelt import (
    GdeltClient,
    GdeltClientError,
    GdeltLongRunSpec,
    GdeltQueryType,
    GkgBulkSpec,
    collect_gdelt_longrun,
    collect_gkg_bulk_extract,
    dry_run_gdelt_longrun,
    gdelt_longrun_status,
)
from mci.market_data import MarketDataSpec, collect_market_data

PROJECT_ROOT

In [ ]:
def make_doc_spec(query_types):
    return GdeltLongRunSpec(
        start_date=START_DATE,
        end_date=END_DATE,
        query_types=tuple(query_types),
        max_records=MAX_RECORDS,
        resume=RESUME,
        overwrite_interim=OVERWRITE_INTERIM,
        request_pause_seconds=REQUEST_PAUSE_SECONDS,
    )


def make_client():
    return GdeltClient(
        max_retries=MAX_RETRIES,
        backoff_seconds=BACKOFF_SECONDS,
        max_backoff_seconds=MAX_BACKOFF_SECONDS,
        jitter_seconds=JITTER_SECONDS,
        request_pause_seconds=REQUEST_PAUSE_SECONDS,
    )


def print_progress(event):
    phase = event.get("phase")
    query_type = event.get("query_type", "")
    day = event.get("date", "")
    if phase == "retry":
        status = event.get("status_code") or event.get("error")
        print(f"{day} {query_type} retry {event.get('attempt')} after {status}; sleeping {event.get('sleep_seconds')}s")
    elif phase == "skip":
        print(f"{day} {query_type} skipped existing checkpoint")
    elif phase == "checkpoint":
        print(f"{day} {query_type} checkpoint saved")
    elif phase == "fatal_failure":
        print(f"{day} {query_type} fatal failure: {event}")
    elif phase == "aggregate":
        print(f"{query_type} aggregate saved: {event.get('article_count')} articles")
    elif phase and phase.startswith("gkg"):
        print(event)


def print_doc_result(result):
    for query_type, raw_path in result.raw_paths.items():
        print(f"{query_type}")
        print(f"  Raw JSON: {raw_path}")
        print(f"  Interim CSV: {result.interim_paths[query_type]}")
        print(f"  Articles: {result.article_counts[query_type]}")

## Dry Run And Checkpoint Status

This does not call the network. It shows request counts, existing checkpoints, missing checkpoints, and estimated minimum duration from the configured polite pause.

In [ ]:
doc_spec = make_doc_spec((GdeltQueryType.ALL_MARKET, GdeltQueryType.CANDIDATE_CRITICISM))
dry_run_gdelt_longrun(doc_spec)

In [ ]:
status = gdelt_longrun_status(doc_spec)
coverage = status.groupby("query_type")["checkpoint_exists"].agg(["sum", "count"])
coverage["missing"] = coverage["count"] - coverage["sum"]
coverage

## Collect All US-Market DOC Headlines

Rerunning this cell resumes from daily checkpoints.

In [ ]:
all_market_result = None
if RUN_GDELT:
    all_market_spec = make_doc_spec((GdeltQueryType.ALL_MARKET,))
    try:
        all_market_result = collect_gdelt_longrun(all_market_spec, client=make_client(), progress_callback=print_progress)
        print_doc_result(all_market_result)
    except FileExistsError as exc:
        print("Outputs already exist. Use existing files, change dates, or keep RESUME=True for daily checkpoints. Raw aggregate outputs are never overwritten.")
        print(exc)
    except GdeltClientError as exc:
        print(exc)
        if exc.resume_guidance:
            print("\nResume snippet:\n" + exc.resume_guidance)
else:
    print("Skipped all-market DOC collection because RUN_GDELT is False.")

## Collect Candidate-Criticism DOC Headlines

Rerunning this cell resumes from daily checkpoints.

In [ ]:
candidate_result = None
if RUN_GDELT:
    candidate_spec = make_doc_spec((GdeltQueryType.CANDIDATE_CRITICISM,))
    try:
        candidate_result = collect_gdelt_longrun(candidate_spec, client=make_client(), progress_callback=print_progress)
        print_doc_result(candidate_result)
    except FileExistsError as exc:
        print("Outputs already exist. Use existing files, change dates, or keep RESUME=True for daily checkpoints. Raw aggregate outputs are never overwritten.")
        print(exc)
    except GdeltClientError as exc:
        print(exc)
        if exc.resume_guidance:
            print("\nResume snippet:\n" + exc.resume_guidance)
else:
    print("Skipped candidate-criticism DOC collection because RUN_GDELT is False.")

## Resume Status After Interruption

Run this after a kernel restart or interrupted collection. The first missing date is where the next resumed run starts fetching.

In [ ]:
status = gdelt_longrun_status(doc_spec)
missing = status[~status["checkpoint_exists"]]
print(f"checkpoint coverage: {len(status) - len(missing)}/{len(status)} days complete")
if missing.empty:
    print("All requested DOC checkpoints are complete.")
else:
    print(f"Next missing day: {missing.iloc[0]['date']} ({missing.iloc[0]['query_type']})")
    display(missing.head())

## Collect Market Data

This fetches SPY, QQQ, RSP, and VIX through `collect_market_data`. Raw market data is cached under `data/raw/market/` and is never overwritten.

In [ ]:
market_data_path = None
if RUN_MARKET_DATA:
    market_spec = MarketDataSpec(start_date=START_DATE, end_date=END_DATE)
    try:
        market_data_path = collect_market_data(market_spec)
        print(f"Saved raw market CSV: {market_data_path}")
    except FileExistsError as exc:
        print("Raw market-data output already exists. Use the existing file or change START_DATE/END_DATE. Raw data is never overwritten.")
        print(exc)
else:
    print("Skipped market-data collection because RUN_MARKET_DATA is False.")

## Sanity Checks

In [ ]:
def preview_csv(path: Path, rows: int = 5):
    data = pd.read_csv(path)
    print(f"{path}")
    print(f"Rows: {len(data)}")
    if "date" in data.columns:
        print(f"Date range: {data['date'].min()} to {data['date'].max()}")
    elif "seendate" in data.columns and not data.empty:
        print(f"First seendate: {data['seendate'].iloc[0]}")
    display(data.head(rows))


for result in (all_market_result, candidate_result):
    if result is None:
        continue
    for path in result.interim_paths.values():
        preview_csv(path)

if market_data_path is not None:
    preview_csv(market_data_path)

## Optional Historical GKG Fallback

This is an audit/candidate-extraction fallback, not equivalent to DOC ArticleList headline rows. It streams GKG zip archives, writes filtered extracts plus a manifest, and does not retain full zip archives unless `GKG_KEEP_ARCHIVES=True`.

In [ ]:
RUN_GKG_BULK = False
GKG_KEEP_ARCHIVES = False
GKG_MAX_ARCHIVES = 2  # keep this low for a smoke test

if RUN_GKG_BULK:
    gkg_spec = GkgBulkSpec(
        start_date=START_DATE,
        end_date=END_DATE,
        filter_terms=("stock market", "Wall Street", "S&P 500", "Nasdaq", "bubble", "overvalued"),
        keep_archives=GKG_KEEP_ARCHIVES,
        max_archives=GKG_MAX_ARCHIVES,
    )
    try:
        gkg_result = collect_gkg_bulk_extract(gkg_spec, progress_callback=print_progress)
        print(f"Filtered GKG extract: {gkg_result.extract_path}")
        print(f"Manifest: {gkg_result.manifest_path}")
        print(f"Archives: {gkg_result.archive_count}; matched rows: {gkg_result.matched_row_count}")
    except FileExistsError as exc:
        print("Filtered GKG outputs already exist. Use existing files or change dates. Raw fallback extracts are not overwritten.")
        print(exc)
else:
    print("Skipped optional GKG fallback because RUN_GKG_BULK is False.")

## Next Steps

Use scripts and package functions for the reproducible pipeline. Do not move processing logic into this notebook.

```bash
python scripts/build_annotation_sample.py --candidate-csv data/interim/gdelt_candidate_criticism_20220101_20260531.csv --all-market-csv data/interim/gdelt_all_us_market_20220101_20260531.csv --seed 1
python scripts/build_mci_daily.py --market-csv data/interim/gdelt_all_us_market_20220101_20260531.csv --criticism-csv data/interim/gdelt_candidate_criticism_20220101_20260531.csv --overwrite
python scripts/build_panel_daily.py --prices-csv data/raw/market/market_prices_spy_qqq_rsp_vix_20220101_20260531.csv --overwrite
python scripts/run_mvp_analysis.py --overwrite
```

Before MCI construction, run the cleaning and trading-day alignment steps appropriate for the collected headline files.